In [12]:
import librosa
import numpy as np
import os
from sklearn.model_selection import train_test_split

def extract_features(audio_path, sr=16000, duration=5):
    """Extract MFCC features from audio"""
    # Load audio
    audio, _ = librosa.load(audio_path, sr=sr, duration=duration)
    
    # Pad if too short
    if len(audio) < sr * duration:
        audio = np.pad(audio, (0, sr * duration - len(audio)))
    
    # Extract MFCCs (Mel-frequency cepstral coefficients)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    
    # Normalize
    mfccs = (mfccs - np.mean(mfccs)) / np.std(mfccs)
    
    return mfccs

def prepare_dataset(data_dir):
    """Prepare training dataset"""
    features = []
    labels = []
    
    # Process elephant sounds
    elephant_dir = os.path.join(data_dir, 'elephant')
    for file in os.listdir(elephant_dir):
        if file.endswith('.wav'):
            path = os.path.join(elephant_dir, file)
            feat = extract_features(path)
            features.append(feat)
            labels.append(1)  # Elephant
    
    # Process background sounds
    background_dir = os.path.join(data_dir, 'other')
    for file in os.listdir(background_dir):
        if file.endswith('.wav'):
            path = os.path.join(background_dir, file)
            feat = extract_features(path)
            features.append(feat)
            labels.append(0)  # Background
    
    return np.array(features), np.array(labels)

# Prepare data
X, y = prepare_dataset('audio_dataset')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

In [13]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def create_model(input_shape):
    """Create lightweight CNN for Raspberry Pi"""
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        
        # Convolutional layers
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Dense layers
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')  # Binary classification
    ])
    
    return model

# Reshape data for CNN (add channel dimension)
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

# Create and compile model
model = create_model(X_train.shape[1:])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 40, 157, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 20, 78, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 20, 78, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 20, 78, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 10, 39, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 10, 39, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 10, 39, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 5, 19, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 5, 19, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 12160)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     1,556,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,649,409 (6.29 MB)

 Trainable params: 1,649,409 (6.29 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# Train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=4,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=3)
    ]
)

# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

Epoch 1/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9868 - loss: 0.3003 - val_accuracy: 0.9869 - val_loss: 0.2173 - learning_rate: 0.0010
Epoch 2/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9868 - loss: 0.0999 - val_accuracy: 0.9869 - val_loss: 0.1523 - learning_rate: 0.0010
Epoch 3/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9868 - loss: 0.0997 - val_accuracy: 0.9869 - val_loss: 0.1137 - learning_rate: 0.0010
Epoch 4/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9868 - loss: 0.1050 - val_accuracy: 0.9869 - val_loss: 0.2025 - learning_rate: 0.0010
Epoch 5/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9868 - loss: 0.1150 - val_accuracy: 0.9869 - val_loss: 0.1460 - learning_rate: 0.0010
Epoch 6/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9868 - loss: 0.1106 - val_accuracy: 0.9869 - val_loss: 0.0864 - learning_rate: 0.0010
Epoch 7/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9868 - loss: 0.

In [6]:
# Convert to TensorFlow Lite for better performance
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optimization settings
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

# Convert
tflite_model = converter.convert()

# Save
with open('elephant_detector.tflite', 'wb') as f:
    f.write(tflite_model)

print("Model optimized and saved!")

INFO:tensorflow:Assets written to: /var/folders/rr/d0gz2wj51xg3h4b_s1cxz0bc0000gn/T/tmpqj3iymnh/assets


INFO:tensorflow:Assets written to: /var/folders/rr/d0gz2wj51xg3h4b_s1cxz0bc0000gn/T/tmpqj3iymnh/assets


Saved artifact at '/var/folders/rr/d0gz2wj51xg3h4b_s1cxz0bc0000gn/T/tmpqj3iymnh'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 94, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  5714831568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5714821888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5714832096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5714832800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5714836496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5714825408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5714827168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5715002112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5715003168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5715004752: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1765386938.084905 3091904 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.


Model optimized and saved!


W0000 00:00:1765386938.084921 3091904 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2025-12-10 22:45:38.085854: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/rr/d0gz2wj51xg3h4b_s1cxz0bc0000gn/T/tmpqj3iymnh
2025-12-10 22:45:38.086508: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2025-12-10 22:45:38.086518: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/rr/d0gz2wj51xg3h4b_s1cxz0bc0000gn/T/tmpqj3iymnh
I0000 00:00:1765386938.091541 3091904 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2025-12-10 22:45:38.092281: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2025-12-10 22:45:38.121201: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/rr/d0gz2wj51xg3h4b_s1cxz0bc0000gn/T/tmpqj3iymnh
2025-12-10 22:45:38.129333: I tensorflow/cc/s

In [ ]:
import numpy as np
import librosa
import tensorflow as tf

class ElephantDetector:
    def __init__(self, model_path, sr=16000, duration=5):
        self.sr = sr
        self.duration = duration
        
        # Load TFLite model
        self.interpreter = tf.lite.Interpreter(model_path=model_path)
        self.interpreter.allocate_tensors()
        
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()

    def extract_features_from_file(self, file_path):
        """Load audio file and extract MFCC features"""
        audio, _ = librosa.load(file_path, sr=self.sr)

        # Pad or trim to match duration
        if len(audio) < self.sr * self.duration:
            audio = np.pad(audio, (0, self.sr * self.duration - len(audio)))
        else:
            audio = audio[:self.sr * self.duration]

        # Extract MFCC
        mfccs = librosa.feature.mfcc(y=audio, sr=self.sr, n_mfcc=40)
        mfccs = (mfccs - np.mean(mfccs)) / np.std(mfccs)

        return mfccs[..., np.newaxis]

    def predict_from_file(self, file_path):
        """Predict elephant presence using audio file"""
        features = self.extract_features_from_file(file_path)
        features = features[np.newaxis, ...].astype(np.float32)

        # Run inference
        self.interpreter.set_tensor(self.input_details[0]['index'], features)
        self.interpreter.invoke()
        prediction = self.interpreter.get_tensor(self.output_details[0]['index'])

        return prediction[0][0]


# -------------------------
#     USAGE
# -------------------------

detector = ElephantDetector("elephant_detector.tflite")

audio_file = input("Enter audio file path (.wav): ")

confidence = detector.predict_from_file(audio_file)

print("\nRESULT:")
print("=" * 30)

if confidence > 0.7:
    print(f"🐘 ELEPHANT DETECTED! Confidence: {confidence:.2%}")
else:
    print(f"No elephant. Confidence: {confidence:.2%}")

print("=" * 30)


In [2]:
import numpy as np
import librosa
import tensorflow as tf

class ElephantDetector:
    def __init__(self, model_path, sr=16000, duration=3):
        self.sr = sr
        self.duration = duration
        
        self.interpreter = tf.lite.Interpreter(model_path=model_path)
        self.interpreter.allocate_tensors()
        
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()

    def extract_features_from_file(self, file_path):
        audio, _ = librosa.load(file_path, sr=self.sr)

        if len(audio) < self.sr * self.duration:
            audio = np.pad(audio, (0, self.sr * self.duration - len(audio)))
        else:
            audio = audio[:self.sr * self.duration]

        mfccs = librosa.feature.mfcc(y=audio, sr=self.sr, n_mfcc=40)
        mfccs = (mfccs - np.mean(mfccs)) / np.std(mfccs)

        return mfccs[..., np.newaxis]

    def predict_from_file(self, file_path):
        features = self.extract_features_from_file(file_path)
        features = features[np.newaxis, ...].astype(np.float32)

        self.interpreter.set_tensor(self.input_details[0]['index'], features)
        self.interpreter.invoke()
        prediction = self.interpreter.get_tensor(self.output_details[0]['index'])

        return prediction[0][0]


# ---------------------------------------
#  SET YOUR AUDIO FILE PATH HERE
# ---------------------------------------

audio_file = "audio_dataset/elephant/African elephant trumpets and growls.mp3"   # <--- CHANGE THIS

detector = ElephantDetector("elephant_detector.tflite")

confidence = detector.predict_from_file(audio_file)

print("\nRESULT:")
print("=" * 30)

if confidence > 0.7:
    print(f"🐘 ELEPHANT DETECTED! Confidence: {confidence:.2%}")
else:
    print(f"No elephant. Confidence: {confidence:.2%}")

print("=" * 30)



RESULT:
No elephant. Confidence: 0.03%


/Users/yuvrajbhatkariya/data/VScode.C++/Projects/EleVision/.venv/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [1]:
import sounddevice as sd
import soundfile as sf

def record_audio(duration=10, filename="output.wav", samplerate=16000):
    print("🎙️ Recording started...")

    # Record audio
    recording = sd.rec(int(duration * samplerate),
                       samplerate=samplerate,
                       channels=1,
                       dtype='float32')

    sd.wait()  # Wait until recording is done
    print("✅ Recording finished.")

    # Save the file
    sf.write(filename, recording, samplerate)
    print(f"💾 Audio saved as {filename}")

# Usage
record_audio(10, "my_recording.wav")


🎙️ Recording started...
✅ Recording finished.
💾 Audio saved as my_recording.wav


In [10]:
import numpy as np
import librosa
import tensorflow as tf

class ElephantDetector:
    def __init__(self, model_path, sr=16000, duration=3):
        self.sr = sr
        self.duration = duration
        
        # Load TFLite model
        self.interpreter = tf.lite.Interpreter(model_path=model_path)
        self.interpreter.allocate_tensors()
        
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()
    
    def extract_features(self, audio):
        required_len = self.sr * self.duration
        if len(audio) < required_len:
            audio = np.pad(audio, (0, required_len - len(audio)))
        else:
            audio = audio[:required_len]
        
        mfccs = librosa.feature.mfcc(y=audio, sr=self.sr, n_mfcc=40)
        mfccs = (mfccs - np.mean(mfccs)) / np.std(mfccs)
        
        return mfccs[..., np.newaxis]

    def predict(self, audio):
        features = self.extract_features(audio)
        features = features[np.newaxis, ...].astype(np.float32)

        self.interpreter.set_tensor(self.input_details[0]['index'], features)
        self.interpreter.invoke()
        prediction = self.interpreter.get_tensor(self.output_details[0]['index'])
        
        return float(prediction[0][0])

    def detect(self, file_path):
        """One function: input file → output result"""
        audio, _ = librosa.load(file_path, sr=self.sr)
        confidence = self.predict(audio)

        if confidence > 0.7:
            return f"🐘 Elephant Present (Confidence: {confidence:.2%})"
        else:
            return f"No Elephant (Confidence: {confidence:.2%})"


# ==========================
# Usage (FILE PATH FIXED HERE)
# ==========================
detector = ElephantDetector("elephant_detector.tflite")

# >>> SET YOUR FILE PATH HERE <<<
audio_file = "audio_dataset/elephant/angry-elephant-40916.mp3"

# Run detection
result = detector.detect(audio_file)
print("\nResult:", result)



Result: No Elephant (Confidence: 0.00%)


In [8]:
import numpy as np
import librosa
import tensorflow as tf

class ElephantDetector10s:
    def __init__(self, model_path, sr=16000, duration=10):
        self.sr = sr
        self.duration = duration
        
        # Load TFLite model
        self.interpreter = tf.lite.Interpreter(model_path=model_path)
        self.interpreter.allocate_tensors()
        
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()
    
    def extract_features(self, audio):
        """Extract MFCC features for EXACT 10 seconds of audio"""
        required_len = self.sr * self.duration

        # Pad or trim to fixed length
        if len(audio) < required_len:
            audio = np.pad(audio, (0, required_len - len(audio)))
        else:
            audio = audio[:required_len]

        # MFCC extraction
        mfccs = librosa.feature.mfcc(y=audio, sr=self.sr, n_mfcc=40)

        # Normalize
        mfccs = (mfccs - np.mean(mfccs)) / np.std(mfccs)

        # Add channel dimension
        return mfccs[..., np.newaxis]

    def predict(self, audio):
        """Return elephant probability"""
        features = self.extract_features(audio)
        features = features[np.newaxis, ...].astype(np.float32)

        # Run inference
        self.interpreter.set_tensor(self.input_details[0]['index'], features)
        self.interpreter.invoke()
        prediction = self.interpreter.get_tensor(self.output_details[0]['index'])

        return float(prediction[0][0])

    def detect(self, file_path):
        """Full pipeline: file → prediction text"""
        audio, _ = librosa.load(file_path, sr=self.sr)
        confidence = self.predict(audio)

        if confidence > 0.7:
            return f"🐘 Elephant Present (Confidence: {confidence:.2%})"
        else:
            return f"No Elephant (Confidence: {confidence:.2%})"


# =====================================
# USAGE
# =====================================

detector = ElephantDetector10s("elephant_detector.tflite")

audio_file = "your_10_second_audio.wav"   # <---- REPLACE THIS

result = detector.detect(audio_file)
print(result)


/var/folders/rr/d0gz2wj51xg3h4b_s1cxz0bc0000gn/T/ipykernel_66016/3550459624.py:50: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(file_path, sr=self.sr)
/Users/yuvrajbhatkariya/data/VScode.C++/Projects/EleVision/.venv/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FileNotFoundError: [Errno 2] No such file or directory: 'your_10_second_audio.wav'

In [ ]:
import numpy as np
import sounddevice as sd
import time
from collections import deque

class ContinuousElephantMonitor:
    def __init__(self, model_path, sr=16000, chunk_duration=3, threshold=0.7):
        self.detector = ElephantDetector(model_path, sr, chunk_duration)
        self.sr = sr
        self.chunk_duration = chunk_duration
        self.threshold = threshold
        self.buffer = deque(maxlen=sr * chunk_duration)
    
    def audio_callback(self, indata, frames, time_info, status):
        """Callback for continuous audio stream"""
        if status:
            print(f"Status: {status}")
        
        # Add to buffer
        self.buffer.extend(indata[:, 0])
        
        # Process when buffer is full
        if len(self.buffer) >= self.sr * self.chunk_duration:
            audio = np.array(self.buffer)
            confidence = self.detector.predict(audio)
            
            if confidence > self.threshold:
                print(f"🐘 ELEPHANT DETECTED at {time.strftime('%H:%M:%S')}! "
                      f"Confidence: {confidence:.2%}")
                # Add alert logic here (SMS, email, alarm, etc.)
    
    def start_monitoring(self):
        """Start continuous monitoring"""
        print("Starting continuous elephant monitoring...")
        print("Press Ctrl+C to stop")
        
        with sd.InputStream(
            callback=self.audio_callback,
            channels=1,
            samplerate=self.sr
        ):
            while True:
                time.sleep(0.1)

# Usage
monitor = ContinuousElephantMonitor('elephant_detector.tflite')
monitor.start_monitoring()